# Mesure de transmission de la grosse lentille

https://rubinobs.atlassian.net/wiki/spaces/LTS/pages/50086854/CBP+Calibration+System+Hardware#Lens

Ce notebook fait l'analyse des mesures de transmission de la grosse lentille réalisées dans la salle optique du Vera Rubin en juin 2026.
Ce notebook part des fichiers `run.py`. Il faut donc les avoir déjà générés via le notebook `Data_Reduction_optical_bench.ipynb`


In [ ]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from scipy.interpolate import interp1d
from datetime import datetime

%matplotlib widget

In [ ]:
main_repo = '/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/data/'

In [ ]:
def get_data_run(run, doplot=True):
    """
    Compute and plot charges ratio between sphere and receiver photodiodes for one run

    run : LSSTRun
        LSSTRun object containing the data for the run
    doplot : bool
        If True, the results are plotted
    """

    ### Get data
    wls = run["set_wl"]
    
    Q_sphere = run['pd_charge_per_burst'][wls > 0.]
    Q_receiver = run['calib_charge_per_burst'][wls > 0.]

    Q_sphere_err = run['pd_charge_per_burst_err'][wls > 0.]
    Q_receiver_err = run['calib_charge_per_burst_err'][wls > 0.]

    ratio = np.abs(Q_receiver / Q_sphere)
    ratio_err = np.abs(ratio) * np.sqrt((Q_sphere_err / Q_sphere)**2 + (Q_receiver_err / Q_receiver)**2)

    ### Plot
    if doplot:
        fig, (ax1,ax2,ax3) = plt.subplots(3, sharex="all", figsize=(8,8))

        ax1.errorbar(wls[wls > 0.], Q_sphere, yerr=Q_sphere_err, c='b', marker='.', ls="")
        ax1.set_ylabel('Charges Sphere [C]')
            
        ax2.errorbar(wls[wls > 0.], Q_receiver, yerr=Q_receiver_err, c='r', marker='.', ls="")
        ax2.set_ylabel('Charges Receiver [C]')

        ax3.errorbar(wls[wls > 0.], ratio, yerr=ratio_err, c='indigo', marker='.', ls="")
        ax3.set_ylabel('Ratio Receiver / Sphere [No unit]')
        ax3.set_xlabel('wavelengths [nm]')

        fig.tight_layout()
        plt.show()

    return ratio, ratio_err

def read_Elana_lens_measurements(file_path):
    """
    Read Elana lens measurements from a CSV file and return the data.

    Parameters:
    file_path : str
        Path to the CSV file containing the Elana lens measurements.

    Returns:
    wls, transmission, error : numpy arrays
        wls : Wavelengths (nm)
        transmission : Transmission values (unitless)
        error : Error values (unitless)
    """


    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)

    wls = df['Wavelength'].values
    transmission = df['lens_ratio'].values
    error = df['ratio_error'].values

    return wls, transmission, error

## Lens transmission measured by Elana

In [ ]:
csv_file = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp_avec_LEDs/lens_ratio_calibration_system_2.csv'

wls_elana, transmission_elana, error_elana = read_Elana_lens_measurements(csv_file)

plt.figure(figsize=(6,3))
plt.errorbar(wls_elana, transmission_elana, yerr=error_elana, color='k', marker='.', ls="")
plt.xlabel('Wavelength [nm]')
plt.ylabel('Transmission [No unit]')
plt.title('Lens Elana Transmission')
plt.ylim(0, 1)
plt.show()


## Get the runs done in the optical room of the Rubin observatory

On range tous les runs dans un dictionnaire. La transmission est calculée en faisant le rapport entre 2 mesures, l'une faite avec et l'autre sans la grosse lentille. On crée un dictionnaire qui associe les runs par paire.

In [ ]:
data_dict = {'20260701_171854': ['Solar cell', 'Avec lentille'],
             '20260701_184358': ['Solar cell', 'Avec lentille, de 300 à 400nm seulement'],
             '20260701_193647': ['Solar cell', 'Sans lentille'],

             '20260703_145059': ['Mini lens', 'Sans grosse lentille'],
             '20260703_163504': ['Mini lens', 'Avec grosse lentille'],

             '20260703_190329': ['Mini lens', 'Avec grosse lentille  Le montage a bougé entre les deux runs, donc pas comparable'],
             '20260703_200311': ['Mini lens', 'Sans grosse lentille'],
             '20260706_140844': ['Mini lens', 'Avec grosse lentille'],

             '20260706_154719': ['Mini lens', 'Avec grosse lentille'],
             '20260706_170942': ['Mini lens', 'Avec grosse lentille + mask'],
             '20260706_185224': ['Mini lens', 'Sans grosse lentille'],
             '20260706_195018': ['Mini lens', 'Avec grosse lentille retournée'],
             '20260706_215451': ['Mini lens', 'Avec grosse lentille'],
             '20260706_234151': ['Mini lens', 'Sans grosse lentille, photodiode à la sortie du tube'],
             '20260707_134736': ['Doublet achromatique', 'Avec doublet achromatique, sans grosse lentille'],
             '20260707_152515': ['Doublet achromatique', 'Avec doublet achromatique, avec grosse lentille'],
             '20260707_174705': ['Mini lens', 'Avec grosse lentille + iris ouvert'],
             '20260707_193623': ['Mini lens', 'Avec grosse lentille + iris fermé'],
             '20260707_203859': ['Mini lens', 'Sans grosse lentille + iris fermé'],
             '20260707_213549': ['Mini lens', 'Sans grosse lentille + iris ouvert']
             }

### Association des runs pour le calcul de la transmission de la lentille
### key = avec grosse lentille, value = [sans grosse lentille, description]
ratio_dict = {'20260701_171854': ['20260701_193647', 'Solar cell'],
              '20260701_184358': ['20260701_193647', 'Solar cell de 300 à 400nm'],
              '20260703_163504': ['20260703_145059', 'Mini lens'],
              '20260706_140844': ['20260703_200311', 'Mini lens'],
              '20260706_195018': ['20260706_185224', 'Mini lens - Big lens reversed'],
              '20260707_174705': ['20260707_213549', 'Mini lens - Iris ouvert'],
              '20260707_193623': ['20260707_203859', 'Mini lens - Iris fermé'],
              '20260707_152515': ['20260707_134736', 'Doublet AC']
}


In [ ]:
### On remplie le dictionnaire data_dict avec les longueurs d'onde, les ratios et les erreurs pour chaque run

# Nombre de runs
nruns = len(data_dict.keys())

# Boucle sur les runs
for i, key in enumerate(data_dict.keys()):
    print(f'Run {i}: {key}')
    # Récupère le fichier run.py
    matching_dirs = [d for d in os.listdir(main_repo) if d.startswith(key)]
    data_dir = os.path.join(main_repo, matching_dirs[0])
    run = np.load(data_dir + "/run.npy", allow_pickle=True)

    # Get the wavelengths
    wls_run = run["set_wl"][run["set_wl"] > 0.]

    # Get the ratio and its error
    ratio, ratio_err = get_data_run(run, doplot=False)

    ### Reorder the arrays by wavelength and append to the data_dict
    sort_indices = np.argsort(wls_run)
    data_dict[key].append(wls_run[sort_indices])
    data_dict[key].append(ratio[sort_indices])
    data_dict[key].append(ratio_err[sort_indices])


In [ ]:
## Plot the ratios for all runs
## Here this is the ratio of the receiver photodiode to the sphere photodiode for each run, with error bars.

plt.figure(figsize=(10,5))
colors = plt.cm.tab20(np.linspace(0, 1, len(data_dict)))
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    plt.errorbar(data_dict[key][2], data_dict[key][3], yerr=np.abs(data_dict[key][4]), marker='.', ls="", color=color, label=f'{data_dict[key][0]}: {data_dict[key][1]}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
#plt.legend(fontsize=8)

In [ ]:
### Pour certains runs, les longueurs d'onde ne sont pas uniques (on a répété la mesure), donc on fait la moyenne des ratios pour chaque longueur d'onde
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    print("Run", i)
    if len(data_dict[key][2]) == len(np.unique(data_dict[key][2])):
        print("Wavelengths are unique, no need to average")
        continue
    else:
        wls_set = np.unique(data_dict[key][2])
        avg = [np.mean(data_dict[key][3][data_dict[key][2]==k]) for k in wls_set]
        err_avg = [np.std(data_dict[key][4][data_dict[key][2]==k])/np.sqrt(len(data_dict[key][4][data_dict[key][2]==k])) for k in wls_set]
        # On remplace les valeurs dans le dictionnaire par les valeurs moyennes
        data_dict[key][2] = wls_set
        data_dict[key][3] = np.array(avg)
        data_dict[key][4] = np.array(err_avg)

In [ ]:
## On plot à nouveau les ratios pour tous les runs, mais cette fois avec les longueurs d'onde uniques et les valeurs moyennes
plt.figure(figsize=(10,5))
colors = plt.cm.tab20(np.linspace(0, 1, nruns))
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    print(i)    
    plt.errorbar(data_dict[key][2], data_dict[key][3], yerr=np.abs(data_dict[key][4]), marker='.', ls="", color=color, label=f'{data_dict[key][0]}: {data_dict[key][1]}')

plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend(fontsize=8)

## Lens Transmission

Pour avoir la transmission de la lentille, on fait le rapport entre 2 scan, l'un réalisé avec et l'autre sans la grosse lentille.

In [ ]:
## Rapport entre 2 scans
plt.figure(figsize=(10,5))

# Loop sur ceux avec la grosse lentille
for i, key1 in enumerate(ratio_dict.keys()):
    key2 = ratio_dict[key1][0] # Run sans grosse lentille
    print(f'Run {i}: {key1} / {key2}')
    wls1 = data_dict[key1][2]
    wls2 = data_dict[key2][2]
    ratio1 = data_dict[key1][3]
    ratio2 = data_dict[key2][3]
    ratio_err1 = data_dict[key1][4]
    ratio_err2 = data_dict[key2][4]
    

    commun_wls = np.intersect1d(wls1, wls2)
    print(f"Commun wavelengths between run {key1} and run {key2}: {commun_wls}")

    # Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
    indices1 = np.searchsorted(wls1, commun_wls)
    indices2 = np.searchsorted(wls2, commun_wls)

    commun_ratio1 = ratio1[indices1]
    commun_ratio2 = ratio2[indices2]

    commun_ratio_err1 = ratio_err1[indices1]
    commun_ratio_err2 = ratio_err2[indices2]

    rapport = commun_ratio1 / commun_ratio2
    rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport
    print(rapport_err)

    # On stocke les résultats dans le dictionnaire ratio_dict (longueurs d'onde, rapport, rapport_err)
    ratio_dict[key1].append(commun_wls)
    ratio_dict[key1].append(rapport)
    ratio_dict[key1].append(rapport_err)

    plt.errorbar(commun_wls, rapport, yerr=rapport_err, marker='.', ls="", label=f'{ratio_dict[key1][1]}')

plt.errorbar(wls_elana, transmission_elana, yerr=error_elana, c='k', marker='.', ls="--",  label='Elana measurement')
plt.ylim(0., 1)
plt.ylabel('Lens transmission')
plt.xlabel('wavelengths [nm]')
plt.legend()


   

In [ ]:
# Save the ratio_dict in a pickle file
with open("/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_ratio_dict.pkl", "wb") as f:
    pickle.dump(ratio_dict, f)

# Build the final transmission curve

Ici on sélectionne les meilleurs mesures et on en prend la moyenne et l'écart-type. Ce sera notre mesure finale de la transmission de la lentille. 

In [ ]:
# Read the ratio_dict from the pickle file
# Ce fichier contient toutes les transmissions de lentilles mesurées avec le banc optique tCBP
with open("/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_ratio_dict.pkl", "rb") as f:
    ratio_dict = pickle.load(f)

# Print the keys of the ratio_dict to verify that it was loaded correctly
print(ratio_dict.keys())

# Make a selection :
selected_keys = ['20260703_163504', '20260706_195018', '20260707_174705', '20260707_193623']

# Plot the selected runs
plt.figure(figsize=(10,4))
for key in selected_keys:
    print(f"Selected run: {key}, description: {ratio_dict[key][1]}")
    plt.errorbar(ratio_dict[key][2], ratio_dict[key][3], yerr=ratio_dict[key][4], marker='.', ls="", label=f'{ratio_dict[key][1]}')
plt.errorbar(wls_elana, transmission_elana, yerr=error_elana, c='k', marker='.', ls="--",  label='Elana measurement')
plt.ylim(0., 1)
plt.ylabel('Lens transmission')
plt.xlabel('wavelengths [nm]')
plt.legend()   
plt.grid()

In [ ]:
# Vérifier les wavelengths disponibles pour chaque scan
# Trouver TOUTES les wavelengths uniques de tous les scans

print("Wavelengths disponibles pour chaque scan:")
all_unique_wls = np.array([])
for key in selected_keys:
    wls = ratio_dict[key][2]
    print(f"{key}: {len(wls)} points, de {wls.min():.1f} à {wls.max():.1f} nm")

    all_unique_wls = np.union1d(all_unique_wls, ratio_dict[key][2])

print(f"\nToutes les wavelengths uniques: {len(all_unique_wls)} points, de {all_unique_wls.min():.1f} à {all_unique_wls.max():.1f} nm")

In [ ]:
### Calcul de la transmission moyenne

# Pour chaque wavelength, compter combien de scans l'ont et la moyenner
avg_ratio = []
avg_ratio_err = []
count_per_wl = []

for wl in all_unique_wls:
    ratios = []
    ratio_errs = []
    
    for key in selected_keys:
        wls = ratio_dict[key][2]
        ratio = ratio_dict[key][3]
        ratio_err = ratio_dict[key][4]
        
        # Chercher si cette wavelength existe dans ce scan
        idx = np.where(np.isclose(wls, wl, rtol=1e-9))[0]
        if len(idx) > 0:
            ratios.append(ratio[idx[0]])
            ratio_errs.append(ratio_err[idx[0]])
    
    count = len(ratios)
    if count > 0:
        avg_ratio.append(np.mean(ratios))
        
        if count > 1:
            # Calculer le STD sur les scans
            err = np.std(ratios, ddof=1) / np.sqrt(count)
        else:
            # Un seul scan : garder l'erreur du dictionnaire
            err = ratio_errs[0]
        
        avg_ratio_err.append(err)
        count_per_wl.append(count)

avg_ratio = np.array(avg_ratio)
avg_ratio_err = np.array(avg_ratio_err)
count_per_wl = np.array(count_per_wl)

print(f"\nMoyenne complète: {len(avg_ratio)} points")
print(f"  - {np.sum(count_per_wl == 4)} points avec 4 scans")
print(f"  - {np.sum(count_per_wl == 3)} points avec 3 scans")
print(f"  - {np.sum(count_per_wl == 2)} points avec 2 scans")
print(f"  - {np.sum(count_per_wl == 1)} point(s) avec 1 scan")
print(f"Ratio moyen: {avg_ratio.min():.3f} à {avg_ratio.max():.3f}")


In [ ]:

# Tracer
plt.figure(figsize=(12,6))
for key in selected_keys:
    wls = ratio_dict[key][2]
    ratio = ratio_dict[key][3]
    ratio_err = ratio_dict[key][4]
    plt.errorbar(wls, ratio, yerr=ratio_err, marker='.', ls="", alpha=0.5, label=key)

plt.errorbar(all_unique_wls, avg_ratio, yerr=avg_ratio_err, marker='.', ls="", markersize=5, color='k', label=f'Moyenne')

plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio")
plt.legend(fontsize=8)
plt.grid()
plt.show()

In [ ]:
# Save the average ratio and its error in a npy file
np.save(f"/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_avg_ratio_{datetime.now().strftime('%Y-%m-%d')}.npy", np.array([all_unique_wls, avg_ratio, avg_ratio_err], dtype=object))

## Ratio entre les mesures 
Pour voir si elles diffèrent juste d'un facteur

!!! TODO : voir si on peut exploiter ça pour réduire encore les barres d'erreur sur la mesure combinée

In [ ]:
key1 = selected_keys[0]
key2 = selected_keys[1]

# Tracer

wls1 = ratio_dict[key1][2]
ratio1 = ratio_dict[key1][3]
ratio_err1 = ratio_dict[key1][4]

wls2 = ratio_dict[key2][2]
ratio2 = ratio_dict[key2][3]
ratio_err2 = ratio_dict[key2][4]

wls_common = np.intersect1d(wls1, wls2)

rapport = []
rapport_err = []
for wl in wls_common:
    idx1 = np.where(np.isclose(wls1, wl, rtol=1e-9))[0]
    idx2 = np.where(np.isclose(wls2, wl, rtol=1e-9))[0]
    
    if len(idx1) > 0 and len(idx2) > 0:
        r1 = ratio1[idx1[0]]
        r2 = ratio2[idx2[0]]
        e1 = ratio_err1[idx1[0]]
        e2 = ratio_err2[idx2[0]]
        
        rapport.append(r1 / r2)
        rapport_err.append(np.sqrt((e1 / r1)**2 + (e2 / r2)**2) * (r1 / r2))

In [ ]:
plt.figure(figsize=(12,6))
plt.errorbar(wls_common, rapport, yerr=rapport_err, marker='.', ls="", markersize=5, color='k', label=f'Rapport')
plt.xlabel("Wavelength [nm]")
plt.ylabel("Transmission ratio")
plt.ylim(0.95, 1.05)
plt.legend(fontsize=8)
plt.grid()
plt.show()